# Nubzuki standing — Colab (GPU)

Trains the same `standing` branch this repo runs on the Mac, on a CUDA GPU.

**Set the runtime to a GPU first**: Runtime → Change runtime type → T4 (or better).

Colab disconnects without warning and wipes its disk when it does, so this
notebook keeps checkpoints on Google Drive and every training cell resumes from
the newest one. Re-running a cell after a disconnect continues where it stopped.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo "No GPU. Runtime > Change runtime type > T4, then rerun."

## 2. Mount Drive

Checkpoints go here so a disconnect costs at most one checkpoint interval.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
RUNS = Path('/content/drive/MyDrive/nubzuki/runs/standing')
RUNS.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', RUNS)

## 3. Get the code

Public repo, so no token is needed. Re-running this pulls the latest commit
rather than cloning again.

In [ ]:
import os
from pathlib import Path

REPO = Path('/content/Robot_Nubzuki')
if REPO.exists():
    !cd {REPO} && git fetch --quiet origin standing && git checkout --quiet standing && git pull --quiet
else:
    !git clone --quiet --branch standing https://github.com/sungwon1ee/Robot_Nubzuki.git {REPO}
os.chdir(REPO)
!git log --oneline -1

## 4. Install

The `gpu` extra pulls the CUDA JAX wheels instead of the CPU ones. This takes a
few minutes, and the "session must restart" prompt is expected — accept it, then
carry on from the next cell, which re-mounts Drive and re-enters the repo.

In [ ]:
!pip install -q -e '.[gpu,test]'

In [ ]:
# Safe to re-run after the install prompts for a restart: it redefines
# everything the later cells need.
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/Robot_Nubzuki')
RUNS = Path('/content/drive/MyDrive/nubzuki/runs/standing')
RUNS.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)

import jax
print('jax', jax.__version__)
print('backend', jax.default_backend())
print('devices', jax.devices())
assert jax.default_backend() == 'gpu', 'Still on CPU: check the runtime type, then reinstall.'

## 5. Verify before spending GPU time

`validate` checks the observation/action contract and the NumPy behaviour Brax
needs. `smoke` runs the whole pipeline — training, checkpoint, ONNX export and
its parity check — in about a thousand steps. Both are cheap and both have
caught real breakage in this project.

In [ ]:
!python -m playground.nubzuki.cli validate
!python -m pytest tests -q

In [ ]:
!python -m playground.nubzuki.cli train --preset smoke --output /content/runs/smoke

## 6. Measure this GPU

Picks an environment count and reports steady-state throughput with the compile
pass excluded. The number is rollout only, so real training is slower — but it
tells you the order of magnitude, and how it compares to 2,219 steps/s on the Mac.

In [ ]:
!python -m playground.nubzuki.cli benchmark --output .local/device_profile.json

## 7. Train

`--num-timesteps` is the total for the whole schedule, not for this run, so a
resumed run finishes the original target instead of restarting it. Raise it later
and rerun this cell to continue past the first target.

`--restore auto` is the default: this cell picks up the newest checkpoint on
Drive by itself. To start over instead, add `--fresh`.

In [ ]:
TOTAL_STEPS = 150_000_000     # the upstream target; lower it or stop early any time
CHECKPOINT_EVERY = 1_000_000  # the most a disconnect can cost

!python -m playground.nubzuki.cli train \
  --preset profile \
  --device-profile .local/device_profile.json \
  --num-timesteps {TOTAL_STEPS} \
  --checkpoint-every {CHECKPOINT_EVERY} \
  --num-eval-envs 128 \
  --output {RUNS}

## 8. Watch

`eval/avg_episode_length` is the readable one: 1000 is a full 20-second episode
without falling. `eval/episode_reward` is near 400 when it survives, since the
alive bonus is +20 per second and dominates every other term.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS}/tensorboard

## 9. Take the policy home

`latest.json` points at the newest checkpoint. The ONNX file and its `policy.json`
must travel together — the runtime refuses a policy whose metadata is missing or
whose calibration hash does not match the robot.

In [ ]:
import json
from pathlib import Path
from google.colab import files

latest = json.loads((RUNS / 'latest.json').read_text())
print(json.dumps(latest, indent=2))
files.download(latest['policy'])
files.download(latest['metadata'])